# 05 · 1D-CNN auf Zeitreihen – vom Signal zur Vorhersage**Von der App zum Code.** In der App[Zeitreihen-CNN-Labor](https://iludis.de/zeitreihen-cnn-labor.html) habt ihr gesehen,was ein **1D-Filter** mit einem Zeitreihen-Signal macht: Er gleitet über die Zeitachseund erzeugt eine **Feature Map** – genau wie der 2D-Filter im[CNN Grayscale Trainer](https://iludis.de/CNN_sim/CNN_sim.html) über ein Bild gleitet.**Der Bogen:** Ihr kennt bereits klassische Zeitreihenverfahren (SARIMAX, Prophet) unddas 2D-CNN für Bilder (`01_basicCNN_didaktisch.ipynb`). Dieses Notebook verbindet beides:dieselbe Faltungs-Idee wie beim 2D-CNN, aber angewendet auf eine **einzelne Zeitachse**statt auf Höhe und Breite eines Bildes.**Datensatz:** [AirPassengers](https://www.kaggle.com/datasets/rakannimer/air-passengers) –monatliche Passagierzahlen 1949–1960, ein Klassiker der Zeitreihenanalyse.---> **Zur Arbeitsweise mit diesem Notebook**>> Es existiert eine ältere Fassung dieses Notebooks. Sie unterscheidet sich an genau> drei Stellen von dieser hier. Eure Aufgabe: die Stellen finden **und begründen**,> warum die Reihenfolge der Arbeitsschritte in dieser Fassung die methodisch richtige> ist. Die Begründung ist wichtiger als der Fund.

## 0 · Konfiguration

In [ ]:
# ============================================================# Konfiguration (zentral & leicht anpassbar)# Alle Stellschrauben an einem Ort -> kein Suchen mehr quer durch den Code.# ============================================================# ReproduzierbarkeitSEED = 42# PfadeDATA_PATH = "Datasets/AirPassengers.csv"# ZeitfensterWINDOW_SIZE = 12     # letzte 12 Monate -> Vorhersage für den 13. MonatHORIZONT    = 1      # wie viele Schritte in die Zukunft wird vorhergesagt# Zeitliche Aufteilung (NICHT zufällig!) - Anteile an den erzeugten FensternTRAIN_ANTEIL = 0.70VAL_ANTEIL   = 0.15# Der Rest (0.15) ist der Testanteil.# Architektur (die eigentlichen Stellschrauben des CNN)N_CHANNELS   = 1     # 1 Kanal: die PassagierzahlCONV_FILTERS = 6     # Filter im Conv1d-BlockKERNEL_SIZE  = 3     # Breite des Faltungskerns (Zeitschritte)POOL_SIZE    = 2     # Reduktionsfaktor durch MaxPool1dFC_HIDDEN    = 32    # Neuronen in der voll verbundenen Schicht# TrainingLEARNING_RATE = 0.01MAX_EPOCHS    = 300GEDULD        = 30   # Early Stopping: Epochen ohne Verbesserung der ValidierungPRINT_EVERY   = 50   # alle N Epochen den Loss ausgeben

In [ ]:
import copyimport torchimport torch.nn as nnimport torch.optim as optimimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltfrom sklearn.preprocessing import MinMaxScalerfrom sklearn.metrics import mean_squared_error, mean_absolute_error# Reproduzierbarkeit: alle sehen dieselben Zufallswerte und Visualisierungentorch.manual_seed(SEED)np.random.seed(SEED)# Haus-StilDUNKELBLAU = "#1B3A5C"MITTELBLAU = "#2E75B6"AKZENT     = "#0969DA"plt.rcParams["font.family"]     = "DejaVu Sans"   # Arial-nah (Arial oft nicht installiert)plt.rcParams["axes.titlecolor"] = DUNKELBLAUplt.rcParams["figure.dpi"]      = 110

## 1 · Daten ansehen – eine Zeitreihe ist eine ZahlenfolgeGenau wie ein Graustufenbild eine Matrix aus Pixelwerten ist, ist eine Zeitreihe eine**Folge von Zahlen entlang der Zeit**. Statt Höhe × Breite haben wir hier nur eineDimension: die Zeit.Zwei Eigenschaften der Reihe werden gleich wichtig: ein deutlicher **Trend** nach obenund eine jährliche **Saisonalität**.

In [ ]:
df = pd.read_csv(DATA_PATH)df["Month"] = pd.to_datetime(df["Month"])passagiere = df["Passengers"].values.astype(np.float32).reshape(-1, 1)print(f"Anzahl Monate: {len(passagiere)}")print("Erste Werte (Passagiere in 1000):", passagiere[:6].flatten())print(f"Wertebereich gesamt: {passagiere.min():.0f} - {passagiere.max():.0f}")plt.figure(figsize=(9, 3))plt.plot(df["Month"], passagiere, color=MITTELBLAU)plt.title("AirPassengers – monatliche Passagierzahlen 1949–1960")plt.xlabel("Jahr"); plt.ylabel("Passagiere (in 1000)")plt.grid(alpha=0.3)plt.tight_layout(); plt.show()

## 2 · Zeitfenster erzeugenEin CNN braucht eine feste Eingabegröße. Beim Bild war das die 8×8-Matrix. Bei derZeitreihe schneiden wir stattdessen gleitende **Fenster** heraus: Die letzten`WINDOW_SIZE` Monate sind die Eingabe, der Monat danach ist das Ziel – dieVorhersage entsteht, indem das Netz aus der Vergangenheit in die Zukunft schaut.Wir arbeiten hier noch mit den **Rohwerten**. Die Skalierung folgt erst später –warum, seht ihr in Abschnitt 4.Wie viele Fenster entstehen? Aus $n$ Messwerten bei Fenstergröße $w$ und Horizont $h$:$$\text{Anzahl Fenster} = n - w - h + 1$$

In [ ]:
def erstelle_fenster(daten, window_size, horizont=1):    '''Zerlegt eine Zeitreihe in überlappende Fenster (X) und die jeweils    "horizont" Schritte später folgenden Zielwerte (y).    Arbeitet auf Rohwerten - es wird hier bewusst nichts skaliert.'''    xs, ys = [], []    for i in range(len(daten) - window_size - horizont + 1):        xs.append(daten[i:i + window_size])        ys.append(daten[i + window_size + horizont - 1])    return np.array(xs, dtype=np.float32), np.array(ys, dtype=np.float32)X_roh, y_roh = erstelle_fenster(passagiere, WINDOW_SIZE, HORIZONT)print(f"Form X_roh: {X_roh.shape}  (Fenster, Zeitschritte, Merkmale)")print(f"Form y_roh: {y_roh.shape}  (Fenster, Zielwert)")print(f"Kontrolle:  {len(passagiere)} - {WINDOW_SIZE} - {HORIZONT} + 1 = "      f"{len(passagiere) - WINDOW_SIZE - HORIZONT + 1}")beispiel = 0plt.figure(figsize=(5, 2))plt.plot(range(WINDOW_SIZE), X_roh[beispiel], "o-", color=MITTELBLAU, label="Eingabefenster")plt.plot(WINDOW_SIZE, y_roh[beispiel], "o", color="red", label="Zielwert")plt.title(f"Ein Trainingsbeispiel (Fenster #{beispiel})")plt.legend(fontsize=8); plt.tight_layout(); plt.show()

## 3 · Aufteilen – **zeitlich**, nicht zufällig, und in **drei** Teile**Erster wichtiger Unterschied zum 2D-CNN:** Bei den Ziffernbildern war`train_test_split` mit zufälliger Mischung völlig in Ordnung – ein Bild einer 3 hatnichts mit dem nächsten zu tun. Bei einer Zeitreihe wäre zufälliges Mischen ein Fehler:Wir würden dem Modell erlauben, aus der **Zukunft** zu lernen, um die **Vergangenheit**vorherzusagen.**Zweiter Unterschied:** Wir teilen in *drei* Blöcke, nicht in zwei.| Menge | Wozu | Wie oft angefasst ||---|---|---|| **Trainingsdaten** | Gewichte anpassen | jede Epoche || **Validierungsdaten** | Early Stopping, Hyperparameter wählen | jede Epoche, aber nie zum Lernen || **Testdaten** | einmalige Schlussbewertung | genau einmal, ganz am Ende |Der Grund für die Dreiteilung: Sobald wir eine Entscheidung *aufgrund* einer Mengetreffen, passen wir uns dieser Menge an. Die Validierungsmenge wird bei jeder Epocheabgefragt und steuert das Early Stopping – sie ist damit Teil des Trainingsprozessesund kann keine ehrliche Schlussnote mehr liefern.

In [ ]:
n_fenster = len(X_roh)n_train   = int(TRAIN_ANTEIL * n_fenster)n_val     = int(VAL_ANTEIL   * n_fenster)n_test    = n_fenster - n_train - n_val# Zeitlich zusammenhängende Blöcke, KEIN MischenX_train_roh, y_train_roh = X_roh[:n_train],                  y_roh[:n_train]X_val_roh,   y_val_roh   = X_roh[n_train:n_train + n_val],   y_roh[n_train:n_train + n_val]X_test_roh,  y_test_roh  = X_roh[n_train + n_val:],          y_roh[n_train + n_val:]# Bis zu welchem Rohindex reicht der Trainingsblock? (letztes Trainingsziel)grenze_train = n_train - 1 + WINDOW_SIZE + HORIZONT - 1print(f"Fenster gesamt: {n_fenster}")print(f"  Training  : {n_train:>3}   Ziele Rohindex "      f"{WINDOW_SIZE + HORIZONT - 1:>3} .. {grenze_train:>3}")print(f"  Validierung: {n_val:>2}   Ziele Rohindex "      f"{grenze_train + 1:>3} .. {grenze_train + n_val:>3}")print(f"  Test      : {n_test:>3}   Ziele Rohindex "      f"{grenze_train + n_val + 1:>3} .. {len(passagiere) - 1:>3}")print("\nIn Monaten:")for name, start, ende in [    ("Training  ", WINDOW_SIZE + HORIZONT - 1, grenze_train),    ("Validierung", grenze_train + 1, grenze_train + n_val),    ("Test      ", grenze_train + n_val + 1, len(passagiere) - 1),]:    print(f"  {name}: {df['Month'][start]:%Y-%m} bis {df['Month'][ende]:%Y-%m}")

### Prüfung der GrenzeDie entscheidende Kontrolle: Berührt irgendein Trainingsfenster einen Zeitpunkt, derzum Validierungs- oder Testziel gehört? Wenn ja, wäre der Split kaputt.Hier greift der Horizont: Bei `HORIZONT = 1` liegt das letzte Trainingsziel genau einenMonat vor dem ersten Validierungsziel. Bei größerem Horizont müsste die Lückeentsprechend breiter sein – rechnet das unten selbst nach.

In [ ]:
hoechster_train_index = n_train - 1 + WINDOW_SIZE + HORIZONT - 1erstes_val_ziel       = n_train + WINDOW_SIZE + HORIZONT - 1print(f"Höchster vom Training berührter Rohindex : {hoechster_train_index}")print(f"Erstes Validierungsziel (Rohindex)       : {erstes_val_ziel}")print(f"Abstand: {erstes_val_ziel - hoechster_train_index} Zeitschritt(e)")assert erstes_val_ziel > hoechster_train_index, "Split überlappt - Leakage!"print("OK - kein Überlapp.")

## 4 · Skalierung – **nach** dem Split, gelernt **nur** auf den TrainingsdatenBeim 2D-CNN haben wir die Pixelwerte durch 16 geteilt. Dieser Divisor war einefeste, bekannte Konstante – da konnte nichts schiefgehen.Beim `MinMaxScaler` ist das anders: Er *lernt* Minimum und Maximum **aus den Daten**.Und gelernte Größen dürfen nur aus dem Trainingsblock stammen. Sonst fließt Wissenüber die Zukunft in die Vorbereitung der Vergangenheit ein.Die Regel lautet in scikit-learn immer gleich:- `fit` bzw. `fit_transform` → **nur** Trainingsdaten- `transform` → alle Mengen, mit den *aus dem Training* gelernten Parametern**Beobachtung, die euch gleich auffallen wird:** Weil die Reihe einen Aufwärtstrend hat,liegen Validierungs- und Testwerte nach der Skalierung **über 1,0**. Das ist keinFehler – das ist die ehrliche Konsequenz. Ein Modell, das im August 1957 gebaut wurde,kennt die Rekordwerte von 1960 nun einmal nicht.

In [ ]:
# Der Scaler sieht ausschließlich den Trainingszeitraum der Rohreihe.scaler = MinMaxScaler(feature_range=(0, 1))scaler.fit(passagiere[:grenze_train + 1])print(f"Scaler gefittet auf Rohindex 0 .. {grenze_train} "      f"({df['Month'][0]:%Y-%m} bis {df['Month'][grenze_train]:%Y-%m})")print(f"  gelerntes Minimum: {scaler.data_min_[0]:.0f}")print(f"  gelerntes Maximum: {scaler.data_max_[0]:.0f}")print(f"  (globales Maximum der Reihe: {passagiere.max():.0f} - liegt im TESTzeitraum "      f"und darf hier nicht auftauchen)")def skaliere(X_roh_block, y_roh_block):    '''Wendet den bereits gefitteten Scaler an - transform, niemals fit.'''    form = X_roh_block.shape    Xs = scaler.transform(X_roh_block.reshape(-1, 1)).reshape(form).astype(np.float32)    ys = scaler.transform(y_roh_block).astype(np.float32)    return Xs, ysX_train, y_train = skaliere(X_train_roh, y_train_roh)X_val,   y_val   = skaliere(X_val_roh,   y_val_roh)X_test,  y_test  = skaliere(X_test_roh,  y_test_roh)print(f"\nWertebereich Training   : {X_train.min():.2f} bis {X_train.max():.2f}")print(f"Wertebereich Validierung: {X_val.min():.2f} bis {X_val.max():.2f}")print(f"Wertebereich Test       : {X_test.min():.2f} bis {X_test.max():.2f}  <- über 1.0, wie erwartet")

## 5 · Umwandlung in TensorenEin 1D-CNN erwartet die Form **(Fenster, Kanäle, Zeitschritte)** – analog zu**(Bilder, Kanal, Höhe, Breite)** beim 2D-CNN. Wir haben 1 Kanal (die Passagierzahl)und müssen die Achsen dafür vertauschen (`permute`).

In [ ]:
def zu_tensor(Xb, yb):    return torch.from_numpy(Xb).permute(0, 2, 1), torch.from_numpy(yb)X_train, y_train = zu_tensor(X_train, y_train)X_val,   y_val   = zu_tensor(X_val,   y_val)X_test,  y_test  = zu_tensor(X_test,  y_test)print(f"Form X_train: {tuple(X_train.shape)}  (Fenster, Kanal, Zeitschritte)")print(f"Form X_val  : {tuple(X_val.shape)}")print(f"Form X_test : {tuple(X_test.shape)}")

## 6 · Das 1D-CNN als `nn.Sequential`Derselbe Aufbau wie beim 2D-CNN – nur mit `Conv1d` und `MaxPool1d` statt `Conv2d` und`MaxPool2d`. Der Filter gleitet nicht mehr über Höhe *und* Breite, sondern nur nochüber die **Zeitachse**.Die Ausgabelänge einer Faltung berechnet sich als$$L_{\text{aus}} = \frac{L_{\text{ein}} - k + 2p}{s} + 1$$mit Kernelgröße $k$, Padding $p$ und Stride $s$. Mit $k=3$, $p=1$, $s=1$ bleibt dieLänge erhalten. Rechnet die Zahlen unten mit.

In [ ]:
# Ausgabelänge der Faltung (Stride 1, Padding = KERNEL_SIZE // 2)conv_out_len = (WINDOW_SIZE - KERNEL_SIZE + 2 * (KERNEL_SIZE // 2)) // 1 + 1pooled_len   = conv_out_len // POOL_SIZEflatten_dim  = CONV_FILTERS * pooled_lenprint(f"Eingabe          : {N_CHANNELS} x {WINDOW_SIZE}")print(f"nach Conv1d      : {CONV_FILTERS} x {conv_out_len}")print(f"nach MaxPool1d   : {CONV_FILTERS} x {pooled_len}")print(f"nach Flatten     : {flatten_dim}")net = nn.Sequential(    nn.Conv1d(N_CHANNELS, CONV_FILTERS,              kernel_size=KERNEL_SIZE, padding=KERNEL_SIZE // 2),   # 0    nn.ReLU(),                                                      # 1    nn.MaxPool1d(POOL_SIZE),                                        # 2    nn.Flatten(),                                                   # 3    nn.Linear(flatten_dim, FC_HIDDEN),                              # 4    nn.ReLU(),                                                      # 5    nn.Linear(FC_HIDDEN, 1),                                        # 6: 1 Vorhersagewert)print()print(net)total = sum(p.numel() for p in net.parameters() if p.requires_grad)print(f"\nTrainierbare Parameter insgesamt: {total}")print(f"  Conv1d: {CONV_FILTERS}*{N_CHANNELS}*{KERNEL_SIZE} + {CONV_FILTERS} "      f"= {CONV_FILTERS * N_CHANNELS * KERNEL_SIZE + CONV_FILTERS}")print(f"  Linear1: {flatten_dim}*{FC_HIDDEN} + {FC_HIDDEN} "      f"= {flatten_dim * FC_HIDDEN + FC_HIDDEN}")print(f"  Linear2: {FC_HIDDEN}*1 + 1 = {FC_HIDDEN + 1}")# Conv-Layer ist Element 0. Anfangs-Kernel sichern (Kopie!) für Vorher/Nachher.kernel_vorher = net[0].weight.detach().clone()

## 7 · Tensor-Formen Schritt für Schritt verfolgenWie beim 2D-CNN: ein Fenster durch das Netz schicken und nach jeder Stufe die Formprüfen. Ein `nn.Sequential` ist auch hier **schneidbar**: `net[:k]`.

In [ ]:
beispiel_fenster = X_train[0:1]   # Form (1, 1, 12): ein einzelnes Fensterwith torch.no_grad():    nach_conv = net[:1](beispiel_fenster)    nach_relu = net[:2](beispiel_fenster)    nach_pool = net[:3](beispiel_fenster)print(f"{'Stufe':<24}{'Form (Kanal x Zeitschritte)'}")print("-" * 52)print(f"{'Eingabe':<24}{tuple(beispiel_fenster.shape[1:])}")print(f"{'Faltung (Conv1d)':<24}{tuple(nach_conv.shape[1:])}")print(f"{'nach ReLU':<24}{tuple(nach_relu.shape[1:])}")print(f"{'nach MaxPool':<24}{tuple(nach_pool.shape[1:])}")print(f"{'flach (Flatten)':<24}{(flatten_dim,)}")print(f"{'Ausgabe (1 Wert)':<24}{(1,)}")

## 8 · Training mit Early Stopping auf den Validierungsdaten**Dritter wichtiger Unterschied zur alten Fassung:** Statt einer festen Epochenzahlzu vertrauen, beobachten wir in jeder Epoche den Verlust auf den **Validierungsdaten**und behalten die Gewichte des besten Zeitpunkts.Das ist genau die Situation, in der man Overfitting *sieht*: Der Trainingsverlustsinkt weiter, der Validierungsverlust steigt wieder. Der Punkt, an dem sich die beidenKurven trennen, ist der Moment, in dem das Netz beginnt, Einzelfälle auswendig zulernen statt Muster.Die Testdaten kommen hier **nicht** vor. Sie bleiben unberührt bis Abschnitt 9.

In [ ]:
criterion = nn.MSELoss()optimizer = optim.Adam(net.parameters(), lr=LEARNING_RATE)verlauf_train, verlauf_val = [], []bester_val   = float("inf")beste_epoche = 0beste_gewichte = copy.deepcopy(net.state_dict())for epoch in range(MAX_EPOCHS):    # --- Trainingsschritt ---    net.train()    optimizer.zero_grad()    loss = criterion(net(X_train), y_train)    loss.backward()    optimizer.step()    verlauf_train.append(loss.item())    # --- Validierung: nur bewerten, nicht lernen ---    net.eval()    with torch.no_grad():        val_loss = criterion(net(X_val), y_val).item()    verlauf_val.append(val_loss)    if val_loss < bester_val:        bester_val, beste_epoche = val_loss, epoch        beste_gewichte = copy.deepcopy(net.state_dict())    if (epoch + 1) % PRINT_EVERY == 0:        print(f"Epoche {epoch + 1:>3}/{MAX_EPOCHS} | "              f"Train {loss.item():.5f} | Val {val_loss:.5f}")    if epoch - beste_epoche >= GEDULD:        print(f"\nEarly Stopping in Epoche {epoch + 1}: "              f"seit {GEDULD} Epochen keine Verbesserung.")        breaknet.load_state_dict(beste_gewichte)print(f"Bestes Modell stammt aus Epoche {beste_epoche + 1} (Val-Loss {bester_val:.5f}).")

In [ ]:
plt.figure(figsize=(6, 3))plt.plot(verlauf_train, color=DUNKELBLAU, label="Training")plt.plot(verlauf_val,   color=AKZENT,     label="Validierung")plt.axvline(beste_epoche, color="gray", linestyle=":", label="gewählte Epoche")plt.title("Verlustkurven – wo trennen sich Training und Validierung?")plt.xlabel("Epoche"); plt.ylabel("MSE-Verlust (normiert)")plt.legend(fontsize=8); plt.grid(alpha=0.3)plt.tight_layout(); plt.show()

## 9 · Evaluation – **Fehlermaß statt Trefferquote****Unterschied zum 2D-CNN:** Bei den Ziffern gab es eine `accuracy_score`(richtig/falsch). Eine Passagierzahl-Vorhersage ist aber keine Klasse, sondern einPunkt auf einer Skala. Wir messen deshalb den durchschnittlichen Fehler in der**Originalskala** (Passagiere), nicht in der normierten Skala.Wir berichten Validierung **und** Test. Die Lücke zwischen beiden ist selbst eineInformation: Ist der Testfehler deutlich größer, haben wir uns beim Early Stoppingan die Validierungsmenge angepasst.

In [ ]:
net.eval()with torch.no_grad():    pred_val_norm  = net(X_val)    pred_test_norm = net(X_test)# Rück-Normalisierung in die Originalskala (Passagierzahlen)pred_val   = scaler.inverse_transform(pred_val_norm.numpy())pred_test  = scaler.inverse_transform(pred_test_norm.numpy())wahr_val   = y_val_rohwahr_test  = y_test_rohfor name, wahr, pred in [("Validierung", wahr_val, pred_val),                         ("Test       ", wahr_test, pred_test)]:    mae  = mean_absolute_error(wahr, pred)    rmse = np.sqrt(mean_squared_error(wahr, pred))    print(f"{name}: MAE {mae:6.1f} Passagiere | RMSE {rmse:6.1f}")

## 10 · Die Pipeline sichtbar machenJetzt das Herzstück – wie in Abschnitt 6 des 2D-Notebooks, nur als **Linienplots**statt Bilder: Wir schicken **ein** Test-Fenster durchs trainierte Netz und zeigenjede Stufe.**Eingabe → Filter → Faltung → ReLU → MaxPool**

In [ ]:
def zeige_feature_maps(tensor, titel, farbe=MITTELBLAU):    '''Zeigt alle Kanäle (Feature Maps) eines 1D-Tensors (1, K, T) als Linienplots.'''    n = tensor.shape[1]    fig, axs = plt.subplots(1, n, figsize=(1.6 * n, 1.6), sharey=False)    for i in range(n):        axs[i].plot(tensor[0, i], color=farbe)        axs[i].set_title(f"#{i}", fontsize=8)        axs[i].set_xticks([]); axs[i].set_yticks([])    fig.suptitle(titel, color=DUNKELBLAU)    plt.tight_layout(); plt.show()idx = 0fenster = X_test[idx:idx + 1]net.eval()with torch.no_grad():    logits = net(fenster)    stufen = {        "Faltung (Conv1d)": net[:1](fenster),        "nach ReLU":        net[:2](fenster),        "nach MaxPool":     net[:3](fenster),    }pred_wert = scaler.inverse_transform(logits.numpy())[0, 0]wahr_wert = y_test_roh[idx, 0]plt.figure(figsize=(4, 2))plt.plot(fenster[0, 0], "o-", color=DUNKELBLAU)plt.title(f"Eingabefenster – wahr: {wahr_wert:.0f}, vorhergesagt: {pred_wert:.0f}")plt.tight_layout(); plt.show()zeige_feature_maps(stufen["Faltung (Conv1d)"],                   f"Schritt 1: Feature Maps nach der Faltung (Länge {conv_out_len})")zeige_feature_maps(stufen["nach ReLU"],                   "Schritt 2: nach ReLU – negative Werte sind 0")zeige_feature_maps(stufen["nach MaxPool"],                   f"Schritt 3: nach MaxPool – verkleinert auf Länge {pooled_len}")

## 11 · Was hat das Netz gelernt?### Filter vorher / nachherWie beim 2D-CNN sind die Filter zu Beginn reiner Zufall. Statt 3×3-Bildausschnittenerkennen sie hier **Muster entlang der Zeit** – z. B. einen Anstieg oder einsaisonales Auf und Ab über 3 Monate.

In [ ]:
kernel_nachher = net[0].weight.detach()vmax = max(kernel_vorher.abs().max().item(), kernel_nachher.abs().max().item())fig, axs = plt.subplots(2, CONV_FILTERS, figsize=(12, 2.6), sharey=True)for i in range(CONV_FILTERS):    axs[0, i].plot(kernel_vorher[i, 0],  color=MITTELBLAU); axs[0, i].axis("off")    axs[1, i].plot(kernel_nachher[i, 0], color=DUNKELBLAU); axs[1, i].axis("off")    axs[0, i].set_ylim(-vmax, vmax); axs[1, i].set_ylim(-vmax, vmax)fig.text(0.5, 0.98, f"Filter vor und nach dem Training (je {KERNEL_SIZE} Zeitschritte)",         ha="center", color=DUNKELBLAU, fontsize=12)fig.text(0.085, 0.70, "vorher",  va="center", ha="right", color=DUNKELBLAU)fig.text(0.085, 0.28, "nachher", va="center", ha="right", color=DUNKELBLAU)plt.tight_layout(rect=[0.10, 0, 1, 0.92]); plt.show()

## 12 · Vorhersage & ehrlicher VergleichDer entscheidende Plot: die komplette Reihe mit echten Werten, darübergelegt dieModellvorhersage **nur auf den Testdaten**, die das Netz nie gesehen hat.Die Trennlinien markieren Validierungs- und Testbeginn.

In [ ]:
achse_val  = np.arange(grenze_train + 1, grenze_train + 1 + n_val)achse_test = np.arange(grenze_train + 1 + n_val, len(passagiere))plt.figure(figsize=(10, 4))plt.plot(np.arange(len(passagiere)), passagiere, color=MITTELBLAU,         label="Echte Werte (gesamter Datensatz)")plt.plot(achse_val,  pred_val,  "--", color="#F5A623", label="Vorhersage (Validierung)")plt.plot(achse_test, pred_test, "--", color="red",     label="Vorhersage (Test)")plt.axvline(grenze_train + 0.5,         color="gray", linestyle=":", label="Start Validierung")plt.axvline(grenze_train + n_val + 0.5, color="black", linestyle=":", label="Start Test")plt.title("1D-CNN – Vorhersage auf ungesehenen Daten")plt.xlabel("Zeitindex (Monate)"); plt.ylabel("Passagiere (in 1000)")plt.legend(fontsize=8); plt.grid(alpha=0.3)plt.tight_layout(); plt.show()

### Schlägt das CNN die naiven Vergleichsmodelle?Ein Modell ist erst dann etwas wert, wenn es einen trivialen Vergleichsmaßstab schlägt.Bei Zeitreihen sind das zwei:- **Persistenz** („morgen = heute"): $\hat{y}[t] = y[t-1]$- **Seasonal Naive** („gleicher Monat wie im Vorjahr"): $\hat{y}[t] = y[t-12]$Bei einer Reihe mit klarer Jahressaison ist Seasonal Naive der *faire* Maßstab.Ein Modell, das nur die Persistenz schlägt, hat unter Umständen nur den Trend gelernt.

In [ ]:
# Zeitindizes der Test-Zielwerte in der Originalreihetest_idx = np.arange(grenze_train + 1 + n_val, len(passagiere))y_true   = passagiere[test_idx].flatten()cnn_pred = pred_test.flatten()naive_pred  = passagiere[test_idx - 1].flatten()     # Persistenz:     y[t] = y[t-1]saison_pred = passagiere[test_idx - 12].flatten()    # Seasonal Naive: y[t] = y[t-12]mae_cnn    = mean_absolute_error(y_true, cnn_pred)mae_naive  = mean_absolute_error(y_true, naive_pred)mae_saison = mean_absolute_error(y_true, saison_pred)def verbesserung(basis, modell):    return (basis - modell) / basis * 100print(f"{'Modell':<22}{'MAE (Passagiere)':>18}{'vs. naiv':>12}")print("-" * 52)print(f"{'naive Baseline':<22}{mae_naive:>18.1f}{'—':>12}")print(f"{'Seasonal Naive':<22}{mae_saison:>18.1f}{verbesserung(mae_naive, mae_saison):>11.1f}%")print(f"{'1D-CNN':<22}{mae_cnn:>18.1f}{verbesserung(mae_naive, mae_cnn):>11.1f}%")print("-" * 52)if mae_cnn < mae_saison:    print("→ Das CNN schlägt auch den fairen Seasonal-Naive-Vergleich.")elif mae_cnn < mae_naive:    print("→ Das CNN schlägt die simple Persistenz, aber NICHT den Seasonal Naive.")else:    print("→ Achtung: Das CNN schlägt nicht einmal die simple Persistenz.")monate_test = df["Month"].values[test_idx]plt.figure(figsize=(10, 4))plt.plot(monate_test, y_true,      color="#444444",  linewidth=2, label="Wahrheit")plt.plot(monate_test, cnn_pred,    color=MITTELBLAU, linewidth=2, label=f"1D-CNN (MAE {mae_cnn:.0f})")plt.plot(monate_test, naive_pred,  color="#9AA0AD",  linestyle="--", label=f"naiv: y[t-1] (MAE {mae_naive:.0f})")plt.plot(monate_test, saison_pred, color="#F5A623",  linestyle=":", linewidth=2, label=f"Seasonal Naive: y[t-12] (MAE {mae_saison:.0f})")plt.title("Vergleich auf den Testdaten – schlägt das CNN die Baselines?")plt.xlabel("Monat"); plt.ylabel("Passagiere (in 1000)")plt.legend(fontsize=8); plt.grid(alpha=0.3)plt.tight_layout(); plt.show()

## 13 · ArbeitsauftragVergleicht dieses Notebook mit der älteren Fassung. Drei Dinge sind anders.Für jeden Fund beantwortet ihr schriftlich:1. **Was** genau wurde geändert? (Zeile bzw. Abschnitt benennen)2. **Welche Information** hätte in der alten Fassung an eine Stelle gelangen können,   an die sie nicht gehört?3. **In welche Richtung** verzerrt der Fehler das gemeldete Ergebnis – zu optimistisch   oder zu pessimistisch? Begründet das.4. Wie müsste man vorgehen, wenn der **Horizont** nicht 1, sondern 3 Monate wäre?Zusatzfrage zum Nachrechnen: Bei `WINDOW_SIZE = 12` und `HORIZONT = 1` entstehen aus144 Monatswerten wie viele Fenster? Wie viele wären es bei `HORIZONT = 3`?